In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# ============================================================
# PROBLEM 12.12.4 — FIR Design by the Frequency Sampling Method
# ============================================================

N = 15
L = (N - 1) // 2
k = np.arange(L + 1)
sample_magnitudes = np.array([1.0, 1.0, 1.0, 1.0, 0.4, 0.0, 0.0, 0.0], dtype=float)

def f3(x):
    x = 0.0 if abs(x) < 5e-4 else x
    return f"{x:.3f}"

def make_table_html(title, col_labels, row_labels, data):
    head = "".join([f"<th>{c}</th>" for c in col_labels])
    rows = ""
    for rlab, row in zip(row_labels, data):
        row_html = "".join([f"<td>{v}</td>" for v in row])
        rows += f"<tr><th>{rlab}</th>{row_html}</tr>"
    return f"""
    <div class="fs-box">
        <div class="fs-title">{title}</div>
        <table class="fs-table">
            <thead><tr><th></th>{head}</tr></thead>
            <tbody>{rows}</tbody>
        </table>
    </div>
    """

# ------------------------------------------------------------
# 8x8 linear system A h = b
# ------------------------------------------------------------

A = np.zeros((L + 1, L + 1), dtype=float)
for kk in range(L + 1):
    for n0 in range(L):
        A[kk, n0] = 2.0 * np.cos((2.0 * np.pi * kk / N) * (L - n0))
    A[kk, L] = 1.0

b = sample_magnitudes.copy()
h_ind = np.linalg.solve(A, b)
h = np.concatenate((h_ind, h_ind[-2::-1]))
n = np.arange(N)

# ------------------------------------------------------------
# Verification at prescribed samples
# ------------------------------------------------------------

omega_k = 2.0 * np.pi * k / N
Hk = np.fft.fft(h, N)
mag_k = np.abs(Hk[:L + 1])

# ------------------------------------------------------------
# Dense DTFT for plots
# ------------------------------------------------------------

omega = np.linspace(0.0, np.pi, 2000)
H = np.exp(-1j * np.outer(omega, n)) @ h
mag = np.abs(H)
mag_db = 20.0 * np.log10(np.maximum(mag, 1e-6))

# ------------------------------------------------------------
# Tables
# ------------------------------------------------------------

A_cols = [f"h[{i}]" for i in range(L + 1)]
A_rows = [f"k={i}" for i in range(L + 1)]
A_data = [[f3(v) for v in row] for row in A]

b_cols = ["value"]
b_rows = [f"b[{i}]" for i in range(L + 1)]
b_data = [[f3(v)] for v in b]

ind_cols = ["Coefficient", "Value", "Coefficient", "Value"]
ind_rows = ["", "", "", ""]
ind_data = []
for i in range(0, L + 1, 2):
    left_name, left_val = f"h[{i}]", f3(h_ind[i])
    if i + 1 <= L:
        right_name, right_val = f"h[{i+1}]", f3(h_ind[i + 1])
    else:
        right_name, right_val = "", ""
    ind_data.append([left_name, left_val, right_name, right_val])

ver_cols = ["Sample", "Value", "Sample", "Value"]
ver_rows = ["", "", "", ""]
ver_data = []
for i in range(0, L + 1, 2):
    left_name, left_val = f"k={i}", f3(mag_k[i])
    if i + 1 <= L:
        right_name, right_val = f"k={i+1}", f3(mag_k[i + 1])
    else:
        right_name, right_val = "", ""
    ver_data.append([left_name, left_val, right_name, right_val])

full_cols = ["Coefficient", "Value", "Coefficient", "Value"]
full_rows = ["", "", "", "", "", "", "", ""]
full_data = []
for i in range(0, N, 2):
    left_name, left_val = f"h[{i}]", f3(h[i])
    if i + 1 < N:
        right_name, right_val = f"h[{i+1}]", f3(h[i + 1])
    else:
        right_name, right_val = "", ""
    full_data.append([left_name, left_val, right_name, right_val])

# ------------------------------------------------------------
# Style
# ------------------------------------------------------------

style = """
<style>
.fs-root { width:980px; max-width:980px; font-family:Arial,sans-serif; }
.fs-header { background:#1f6f9a; color:white; padding:10px 14px; border-radius:6px 6px 0 0; font-size:18px; font-weight:bold; }
.fs-intro { border:1px solid #9fc7da; border-top:none; background:#f4fbff; padding:10px 14px; border-radius:0 0 6px 6px; font-size:13.5px; line-height:1.50; margin-bottom:8px; }
.fs-row { display:flex; gap:8px; align-items:flex-start; margin-bottom:8px; }
.fs-col-72 { width:72%; }
.fs-col-28 { width:28%; }
.fs-col-50 { width:50%; }
.fs-col-100 { width:100%; }
.fs-box { border:1px solid #9fc7da; border-radius:4px; padding:7px; background:white; }
.fs-title { font-size:13.5px; font-weight:bold; color:#0a5f8b; margin-bottom:6px; }
.fs-mini { font-size:12.5px; line-height:1.50; }
.fs-formula { border:1px solid #9fc7da; border-radius:4px; padding:8px 12px; margin:8px 0; font-size:14px; background:#ffffff; }
.fs-table { width:100%; border-collapse:collapse; table-layout:fixed; font-size:11.5px; }
.fs-table th, .fs-table td { border:1px solid #8eb8cc; padding:4px 5px; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
.fs-table th { background:#eef7fb; font-weight:bold; text-align:center; }
.fs-table td { text-align:right; }
.fs-table td:first-child, .fs-table th:first-child { text-align:center; }
</style>
"""

# ------------------------------------------------------------
# HTML blocks
# ------------------------------------------------------------

display(HTML(style))

header_html = f"""
<div class="fs-root">
    <div class="fs-header">Problem 12.12.4 — FIR Design by the Frequency Sampling Method</div>
    <div class="fs-intro">
        This notebook reproduces numerically the worked example of the frequency-sampling method for a
        real, symmetric, linear-phase FIR filter of length <b>N = {N}</b> samples. The coefficient
        matrix is formed from the prescribed magnitude samples, the independent coefficients are computed,
        the full impulse response is reconstructed from symmetry, and the resulting magnitude response is
        plotted in both linear and logarithmic scale.
    </div>
    <div class="fs-row">
        <div class="fs-col-50">
            <div class="fs-box fs-mini">
                <div class="fs-title">Given design data</div>
                Filter length: <b>N = {N}</b><br>
                Linear-phase delay: <b>(N-1)/2 = {L}</b> samples<br>
                Symmetry condition: <b>h[n] = h[{N-1}-n]</b>
            </div>
        </div>
        <div class="fs-col-50">
            <div class="fs-box fs-mini">
                <div class="fs-title">Prescribed magnitude samples</div>
                |H<sub>d</sub>(e<sup>j2πk/15</sup>)| = 1 for k = 0,1,2,3<br>
                |H<sub>d</sub>(e<sup>j2πk/15</sup>)| = 0.4 for k = 4<br>
                |H<sub>d</sub>(e<sup>j2πk/15</sup>)| = 0 for k = 5,6,7
            </div>
        </div>
    </div>
    <div class="fs-formula">
        <b>DFT sampling relation:</b> &nbsp; H[k] = H<sub>d</sub>(e<sup>j2πk/N</sup>)
        &nbsp;&nbsp;&nbsp;&nbsp;
        <b>Inverse relation:</b> &nbsp; h[n] = (1/N) &sum; H[k]e<sup>j2πkn/N</sup>
    </div>
</div>
"""
display(HTML(header_html))

tables_html = f"""
<div class="fs-root">
    <div class="fs-row">
        <div class="fs-col-72">{make_table_html("Coefficient matrix A", A_cols, A_rows, A_data)}</div>
        <div class="fs-col-28">{make_table_html("Right-hand side b", b_cols, b_rows, b_data)}</div>
    </div>
    <div class="fs-row">
        <div class="fs-col-50">{make_table_html("Independent coefficients h[0] ... h[7]", ind_cols, ind_rows, ind_data)}</div>
        <div class="fs-col-50">{make_table_html("Verification of prescribed samples", ver_cols, ver_rows, ver_data)}</div>
    </div>
    <div class="fs-row">
        <div class="fs-col-100">{make_table_html("Complete impulse response h[n]", full_cols, full_rows, full_data)}</div>
    </div>
</div>
"""
display(HTML(tables_html))

# ------------------------------------------------------------
# Plots
# ------------------------------------------------------------

fig, ax = plt.subplots(1, 3, figsize=(12.0, 3.7))

markerline, stemlines, baseline = ax[0].stem(n, h, basefmt=' ')
plt.setp(markerline, markersize=4, color='red')
plt.setp(stemlines, linewidth=1.2, color='red')
ax[0].axvline(L, linestyle='--', linewidth=0.9, color='dodgerblue')
ax[0].set_title('FIR Impulse Response', fontsize=12)
ax[0].set_xlabel('Sample index n', fontsize=10)
ax[0].set_ylabel('h[n]', fontsize=10)
ax[0].tick_params(labelsize=9)
ax[0].grid(True, linestyle=':', alpha=0.3)

ax[1].plot(omega / np.pi, mag, color='red', linewidth=1.3, label='Resulting FIR response')
ax[1].plot(2.0 * k / N, sample_magnitudes, 'o', color='dodgerblue', markersize=3, label='Specified samples')
ax[1].set_xlim(0, 1)
ax[1].set_ylim(-0.05, 1.15)
ax[1].set_title('Magnitude Response — Linear Scale', fontsize=12)
ax[1].set_xlabel(r'Normalized frequency $\omega/\pi$', fontsize=10)
ax[1].set_ylabel(r'$|H(e^{j\omega})|$', fontsize=10)
ax[1].tick_params(labelsize=9)
ax[1].grid(True, linestyle=':', alpha=0.3)
ax[1].legend(fontsize=8.5, loc='lower left', frameon=False)

ax[2].plot(omega / np.pi, mag_db, color='red', linewidth=1.3)
ax[2].set_xlim(0, 1)
ax[2].set_ylim(-100, 5)
ax[2].set_title('Magnitude Response — dB Scale', fontsize=12)
ax[2].set_xlabel(r'Normalized frequency $\omega/\pi$', fontsize=10)
ax[2].set_ylabel('Magnitude [dB]', fontsize=10)
ax[2].tick_params(labelsize=9)
ax[2].grid(True, linestyle=':', alpha=0.3)

plt.tight_layout()
plt.show()